**Human Movement Intelligence | Multimodal Movement Learning | Presented by: Reza Saadatyar (2026)**

**02 | Signal Quality, Aligned Windows and Participant Splits**

![Signal Quality, Aligned Windows and Participant Splits workflow](../../images/02_signal_processing_and_aligned_windows.svg)

**Research objective:** Construct consistently labelled, physically paired E1 windows and person-independent Development/Test assignments.

**Dataset:** Official NinaPro DB2 exercise E1, corrected source labels and the Notebook 01 source dictionary.

**Notebook scope:** Inspect each source participant, retain source indices, calculate paired statistical features and establish the common evaluation cohort.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("multimodal_movement_learning")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path
import time

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 6. Study Research Utilities ------------------------
from ninapro_source import ensure_subject_archive, load_exercise, inspect_record
from ninapro_dataset import (
    build_window_registry, select_windows_uniformly, save_source_arrays,
    construct_feature_arrays, assign_participants, check_artifact_alignment,
)

# ---------------------- 7. Source and Output Locations -----------------------
study_dir = root / "multimodal_movement_learning"
output_dir = study_dir / "output"
processed_dir = root / "data" / "processed" / "ninapro_db2"
raw_dir = root / "data" / "raw" / "ninapro_db2"
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


**02. Load the Verified Source Label Contract**

The first notebook supplies the source-defined corrected class list. Fixing this list before model fitting prevents favourable class selection based on Development or Test scores. `restimulus` is the label source; the predictive feature arrays contain only measured sEMG and acceleration.

In [ ]:
# ================ 02. Load the Verified Source Label Contract ================

NOTEBOOK_NUMBER = 2
source_labels = pd.read_csv(output_dir / "01_label_dictionary.csv")
source_audit = pd.read_csv(output_dir / "01_pairing_registry.csv")
classes = np.sort(source_labels.label.to_numpy(int))
if len(classes) < 2 or not np.array_equal(classes, np.unique(classes)):
    raise ValueError("The audited E1 label dictionary has fewer than two valid classes.")
if int(source_audit.loc[0, "exercise_id"]) != 1:
    raise ValueError("Notebook 01 must audit the source exercise E1.")
print("Source-derived target classes:", classes.tolist())


**03. Ingest the First Exercise Across Participants**

The official DB2 population comprises 40 subject archives. Process them sequentially to avoid loading all original recordings into memory simultaneously. The file reader verifies each subject's paired source rows, channels and corrected class IDs; any unexpected class code or missing source causes an explicit error rather than an unreported exclusion.

In [ ]:
# ============= 03. Ingest the First Exercise Across Participants =============

SUBJECT_NUMBERS = tuple(range(1, 41))
DOWNLOAD_MISSING = True
WINDOW_MS = 300
STEP_MS = 100
MAX_WINDOWS_PER_SUBJECT = 1000  # Predeclared, time-uniform, label-blind compute budget.
TEST_PEOPLE = 5
DEVELOPMENT_FOLDS = 5
SPLIT_SEED = 42

source_rows, window_parts, trial_parts = [], [], []
for number in SUBJECT_NUMBERS:
    archive = ensure_subject_archive(raw_dir, number, download=DOWNLOAD_MISSING)
    record = load_exercise(archive, number, exercise=1)
    actual = set(np.unique(record.restimulus).tolist())
    if not actual.issubset(set(classes)):
        raise ValueError(f"Unexpected E1 corrected labels in {record.subject_id}: {actual-set(classes)}")
    window_table, trial_table = build_window_registry(
        record, set(classes), window_ms=WINDOW_MS, step_ms=STEP_MS,
    )
    if window_table.empty:
        raise ValueError(f"No valid paired E1 windows in {record.subject_id}.")
    save_source_arrays(record, processed_dir)
    source_rows.append(inspect_record(record))
    window_parts.append(window_table)
    trial_parts.append(trial_table)
    print(f"{record.subject_id}: {len(window_table):,} eligible source windows")

source_inventory = pd.DataFrame(source_rows)
all_windows = pd.concat(window_parts, ignore_index=True)
all_trials = pd.concat(trial_parts, ignore_index=True)
print("Eligible source participants:", source_inventory.subject_id.nunique())
print("Full source-window inventory:", len(all_windows))


**04. Select Time-Uniform Paired Windows**

A deterministic cap limits deep-learning storage and training cost without using model scores or favourable class outcomes. Selection is uniform over each participant's chronological, source-valid windows. The same retained window keys, labels and physical-time spans are used by all classical, unimodal and fusion models.

In [ ]:
# ================== 04. Select Time-Uniform Paired Windows ===================

index = select_windows_uniformly(all_windows, MAX_WINDOWS_PER_SUBJECT)
observed = (index.groupby(["subject_id", "label"]).size()
            .unstack(fill_value=0).reindex(columns=classes, fill_value=0))
if (observed.sum(axis=0) == 0).any():
    raise ValueError("The predeclared window selection omitted an audited E1 class.")
print("Retained paired windows:", len(index))
display(observed.head())
display(index.head())


**05. Construct Statistical Features on the Same Window Keys**

For each sEMG channel calculate RMS, mean absolute value, waveform length and log variance. For each accelerometer axis calculate mean and standard deviation, and for each triaxial sensor calculate the mean and standard deviation of vector magnitude. The resulting 48 sEMG and 96 acceleration features retain the same source-row order as the lazy neural-window dataset.

In [ ]:
# ======== 05. Construct Statistical Features on the Same Window Keys =========

emg_features, acc_features = construct_feature_arrays(index, processed_dir)
check_artifact_alignment(index, (emg_features, acc_features))
np.save(processed_dir / "02_emg_features.npy", emg_features)
np.save(processed_dir / "02_acc_features.npy", acc_features)
print("Statistical sEMG features:", emg_features.shape)
print("Statistical Acc features:", acc_features.shape)
print("Neural source-window samples:",
      int(index.end_sample.iloc[0] - index.start_sample.iloc[0]))
print("Neural channels: sEMG 12 / acceleration 36")


**06. Freeze Participant-Disjoint Development and Test Roles**

Assign complete participants to reserved Test or Development, then assign Development participants to person-disjoint validation folds. No overlapping windows from one person can appear in both roles. Outer-fold assessment and any inner stopping or scaling use permitted training people only.

In [ ]:
# ======== 06. Freeze Participant-Disjoint Development and Test Roles =========

participant_roles, folds = assign_participants(
    index, test_people=TEST_PEOPLE, folds=DEVELOPMENT_FOLDS,
    seed=SPLIT_SEED,
)
assert not set(participant_roles.loc[participant_roles.role == "Test", "subject_id"]) & set(folds.subject_id)
assert set(index.subject_id) == set(participant_roles.subject_id)
display(participant_roles.groupby("role", as_index=False).agg(people=("subject_id", "nunique")))
display(folds.groupby("validation_fold", as_index=False).agg(people=("subject_id", "nunique")))


**07. Inspect Cohort Balance and Paired Channel Quality**

Participant-level class support and source-channel finite values identify limitations of the selected cohort. These are descriptive counts rather than outcome-driven filters. The figure shows observed source-data completeness for each sensing modality.

In [ ]:
# =========== 07. Inspect Cohort Balance and Paired Channel Quality ===========

quality = pd.DataFrame({
    "modality": ["sEMG", "Acceleration"],
    "mean_finite_fraction": [source_inventory.emg_finite_fraction.mean(),
                             source_inventory.acc_finite_fraction.mean()],
})
display(quality)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].bar([str(v) for v in classes], observed.sum(axis=0).to_numpy())
axes[0].set(title="Retained window support by source class", xlabel="Corrected E1 label", ylabel="Paired windows")
axes[1].bar(quality.modality, quality.mean_finite_fraction)
axes[1].set(title="Original channel completeness", ylabel="Mean finite fraction", ylim=(0, 1.05))
for ax in axes: ax.grid(axis="y", alpha=.2)
save_figure(fig, output_dir / "02_channel_quality.png")
plt.show()


**08. Save Trial, Window and Person-Fold Contracts**

The canonical `window_row` and `window_key` connect the source-trial table, statistical matrices and lazy neural inputs. The two participant assignment tables define all later fold membership. Numerical caches may be reconstructed from source archives if a new Colab kernel does not retain local arrays.

In [ ]:
# ============= 08. Save Trial, Window and Person-Fold Contracts ==============

all_trials.to_csv(output_dir / "02_aligned_trial_index.csv", index=False)
index.to_csv(output_dir / "02_window_index.csv", index=False)
participant_roles.to_csv(output_dir / "02_participant_split_assignments.csv", index=False)
folds.to_csv(output_dir / "02_development_fold_assignments.csv", index=False)
source_inventory.to_csv(output_dir / "02_source_inventory.csv", index=False)
print("Final paired window rows:", len(index))
print("Independent people:", index.subject_id.nunique())


**09. Current Outputs and Scientific Interpretation**

The class dictionary, canonical paired-window index, Development/Test role table and Development fold assignments are the direct numerical interfaces to the model-development notebooks. Statistical scaling and neural normalisation are fitted within permitted Training partitions.

In [ ]:
# ============= 09. Current Outputs and Scientific Interpretation =============

display(pd.DataFrame({"output_file": [
    p.name for p in sorted(output_dir.glob("%02d_*" % NOTEBOOK_NUMBER))
    if p.is_file()
]}))
download_outputs_from_colab(
    folder="multimodal_movement_learning/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
